In [ ]:
"""
03_temporal_split.py
---------------------
Splits each customer's transaction history into:
    - a calibration window (used to build features later)
    - a holdout window (used to build the prediction target later)

A customer is only "eligible" for modeling if their holdout window
has fully completed by OBSERVATION_END - otherwise we'd be trying
to predict a future that hasn't happened yet in the dataset.

Reads:
    data/raw/customers.csv
    data/raw/transactions.csv

Writes:
    data/processed/eligible_customers.csv
    data/processed/calibration_transactions.csv
    data/processed/holdout_transactions.csv
"""

import os
import pandas as pd
from datetime import timedelta

# ----------------------------------------------------------------
# 1. PARAMETERS
# ----------------------------------------------------------------
CALIBRATION_DAYS = 90   # "early window" - what the model gets to see
HOLDOUT_DAYS = 180      # "later window" - what we're trying to predict
OBSERVATION_END = pd.Timestamp("2024-06-30")  # must match 01_data_generation.py

# ----------------------------------------------------------------
# 2. LOAD DATA
# ----------------------------------------------------------------
customers = pd.read_csv("data/raw/customers.csv", parse_dates=["acquisition_date"])
transactions = pd.read_csv("data/raw/transactions.csv", parse_dates=["transaction_date"])

print(f"Total customers: {len(customers)}")

# ----------------------------------------------------------------
# 3. DEFINE EACH CUSTOMER'S WINDOWS (their own personal clock,
#    starting from their own acquisition date - not a shared
#    calendar date)
# ----------------------------------------------------------------
customers["calibration_end_date"] = customers["acquisition_date"] + timedelta(days=CALIBRATION_DAYS)
customers["holdout_end_date"] = customers["acquisition_date"] + timedelta(days=CALIBRATION_DAYS + HOLDOUT_DAYS)

# ----------------------------------------------------------------
# 4. ELIGIBILITY FILTER
#    A customer is only usable if their holdout window has fully
#    played out by the time our dataset ends.
# ----------------------------------------------------------------
customers["eligible"] = customers["holdout_end_date"] <= OBSERVATION_END

eligible_count = customers["eligible"].sum()
excluded_count = len(customers) - eligible_count

print(f"Eligible customers (full calibration + holdout window available): {eligible_count}")
print(f"Excluded customers (acquired too recently to have a full holdout window yet): {excluded_count}")

if excluded_count > 0:
    print("\nExcluded customers' acquisition dates range from:")
    excluded = customers[~customers["eligible"]]
    print(f"  {excluded['acquisition_date'].min().date()} to {excluded['acquisition_date'].max().date()}")
else:
    print("\nNote: 0 customers excluded - this dataset's acquisition window was deliberately")
    print("set with enough buffer before OBSERVATION_END that every customer has a full")
    print("holdout window available. On real-world data (where acquisitions run right up")
    print("to 'today'), this filter typically does exclude a meaningful chunk - it's still")
    print("an important, defensible step to keep in the pipeline.")

eligible_customers = customers[customers["eligible"]].copy()

# ----------------------------------------------------------------
# 5. SPLIT TRANSACTIONS INTO CALIBRATION / HOLDOUT, PER CUSTOMER
# ----------------------------------------------------------------
merged = transactions.merge(
    eligible_customers[["customer_id", "calibration_end_date", "holdout_end_date"]],
    on="customer_id",
    how="inner",  # drops transactions belonging to excluded customers
)

calibration_transactions = merged[
    merged["transaction_date"] <= merged["calibration_end_date"]
][["customer_id", "transaction_date", "amount"]]

holdout_transactions = merged[
    (merged["transaction_date"] > merged["calibration_end_date"]) &
    (merged["transaction_date"] <= merged["holdout_end_date"])
][["customer_id", "transaction_date", "amount"]]

print(f"\nCalibration-window transactions: {len(calibration_transactions)}")
print(f"Holdout-window transactions: {len(holdout_transactions)}")

# Sanity check: every eligible customer should have at least one
# calibration transaction (their acquisition purchase), but not
# necessarily a holdout one (some may have churned by then).
customers_with_holdout_activity = holdout_transactions["customer_id"].nunique()
print(f"Eligible customers with at least one holdout-window purchase: "
      f"{customers_with_holdout_activity} ({customers_with_holdout_activity / eligible_count:.1%})")
print("(The rest had calibration-window activity but went quiet in the holdout window -")
print(" their target value will correctly be 0, which is real, useful signal for the model.)")

# ----------------------------------------------------------------
# 6. SAVE OUTPUTS
# ----------------------------------------------------------------
os.makedirs("data/processed", exist_ok=True)
eligible_customers.to_csv("data/processed/eligible_customers.csv", index=False)
calibration_transactions.to_csv("data/processed/calibration_transactions.csv", index=False)
holdout_transactions.to_csv("data/processed/holdout_transactions.csv", index=False)

print("\nSaved: data/processed/eligible_customers.csv")
print("Saved: data/processed/calibration_transactions.csv")
print("Saved: data/processed/holdout_transactions.csv")